>NOTE: This is notebook is for exploration before creating the final ingestion module, which will be a Python script

## Ingestion Strategy

We will loop over each wdl file in `data/wdl/` and pull the tool name (from the filename) and key metadata from the wdl `meta` section.

**Example WDL task metadata (from `ww-strelka.wdl`)**

```{python}
    topic: "genomics,dna_polymorphism"
    species: "eukaryote"
    operation: "variant_calling"
    input_sample_required: "bam:nucleic_acid_sequence_alignment:bam,bai:data_index:bai"
    input_sample_optional: "target_regions_bed:annotation_track:bed"
    input_reference_required: "ref_fasta:dna_sequence:fasta,ref_fasta_index:data_index:fai"
    input_reference_optional: "none"
    output_sample: "variants_vcf:sequence_variations:vcf,variants_vcf_index:data_index:tbi"
    output_reference: "none"
```

**Example ChromaDB collection metadata for task above**

```{python}
collection.add(
    ids=["id1"],
    documents=["entire wdl task"],
    metadatas=[{
        "tool": strelka,
        "task": "strelka_germline",
        "topic": ["genomics", "dna_polymorphism"],
        "species": ["eukaryote"],
        "operation": "variant_calling",
        "input_sample_data_types": ["nucleic_acid_sequence_alignment", "data_index"],
        "input_sample_format_types": ["bam", "bai"],
        "output_sample_data_types": ["sequence_variations", "data_index"]
    }],
)
```

Note that I'm only including information for required sample inputs and the sample output in the ChromaDB metadata. 

It's possible to search for substrings in the wdl task text itself using the ChromaDB `where_document` filter. This would allow us to create metadata strings from user inputs and search for those in the wdl `meta` section.

## Ingestion Steps

For each wdl:

1. Extract wdl tasks and put into a dictionary
    - Extract the tool name from the file name (e.g. `bowtie` from `ww-bowtie.wdl`)
    - Extract the tasks and their name using regex to 'chunk' by task
2. Parse the wdl task and extract key metadata for ChromaDB
3. Add entire wdl task and key metadata to a ChromaDB collection
4. Confirm we can filter by metadata by evaluating ChromaDB retrieval

In [16]:
import chromadb
import os
import re

## 1. Extract WDL tasks

Get the entire task and some basic metadata along the way (tool name, task name)

In [2]:
def get_tasks(wdl, toolname):
    """Extract wdl task and task name

    Use regex to identify individual wdk tasks and store each
    in a tuple with basic metadata. Put all tasks in the input
    text into a list and return that list.
    """
    tasks = []
    lines = wdl.splitlines(keepends=True)
    i = 0
    while i < len(lines):
        # Look for 'task {' to start the chunk
        match = re.match(r'^task\s+(\w+)\s*\{', lines[i])
        if match:
            task_name = match.group(1)
            depth = 0
            start = i
            while i < len(lines):
                depth += lines[i].count('{') - lines[i].count('}')
                i += 1
                if depth == 0:
                    break
            # Store the entire task and metadata we have so far
            wdl_task = ''.join(lines[start:i])
            wdl_meta = {"tool": toolname, "task": task_name}
            tasks.append((wdl_meta, wdl_task))
        else:
            i += 1
    return tasks



In [46]:
wdl_dir = '../data/wdl/'

metas_tasks = []
for wdl in os.listdir(wdl_dir):
    toolname = wdl[3:-4]
    with open(os.path.join(wdl_dir, wdl), "r") as f:
        content = f.read()
    tasks_list = get_tasks(content,toolname)
    metas_tasks.extend(tasks_list)

In [48]:
metas_tasks[:3]

[({'tool': 'cnvkit', 'task': 'create_reference'},
  'task create_reference {\n  meta {\n    author: "Taylor Firman"\n    email: "tfirman@fredhutch.org"\n    description: "Create CNVkit reference from normal samples or pooled reference"\n    url: "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-cnvkit/ww-cnvkit.wdl"\n    outputs: {\n        reference_cnn: "CNVkit reference file (.cnn)"\n    }\n    topic: "genomics,copy_number_variation"\n    species: "eukaryote"\n    operation: "indexing"\n    input_sample_required: "bam_files:nucleic_acid_sequence_alignment:bam,bam_indices:data_index:bai"\n    input_sample_optional: "target_bed:annotation_track:bed,antitarget_bed:annotation_track:bed"\n    input_reference_required: "reference_fasta:dna_sequence:fasta,reference_fasta_index:data_index:fai"\n    input_reference_optional: "none"\n    output_sample: "none"\n    output_reference: "reference_cnn:data_index:cnn"\n  }\n\n  parameter_meta {\n    bam_files:

## 2. Parse WDL tasks for metadata

In [49]:
def parse_meta(task):
    """Extract ChromaDB metadata from the meta block of a WDL task string."""

    extracted_meta = {}

    # Pull out the metadata section
    meta_match = re.search(r'meta\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', task, re.DOTALL)
    if not meta_match:
        return {}
    meta_block = meta_match.group(1)

    # Metadata that is just comma-separated strings
    for tag in ['topic', 'species', 'operation']:
        full_line = re.search(rf'^\s+{tag}:\s+"([^"]+)"', meta_block, re.MULTILINE)
        extracted_meta[tag] = full_line.group(1).split(',')
    
    # Metadata that is using comma-separated, colon-separated format
    # E.g., ref_fasta:dna_sequence:fasta
    for tag in ['input_sample_required', 'output_sample']:
        full_line = re.search(rf'^\s+{tag}:\s+"([^"]+)"', meta_block, re.MULTILINE)
        param_substrings = [param.split(":") for param in full_line.group(1).split(",")]
        # Some tasks don't take input or produce output, and just have 'none' listed
        if param_substrings[0][0] == 'none':
            param_data = ['none']
            param_format = ['none']
        else:
            param_data = {i[1] for i in param_substrings}
            param_format = {i[2] for i in param_substrings}
        # Now store extracted metadata
        if tag == 'input_sample_required':
            extracted_meta['input_sample_data_types'] = list(param_data)
            extracted_meta['input_sample_format_types'] = list(param_format)
        else:
            extracted_meta['output_sample_data_types'] = list(param_data)
    
    return extracted_meta

In [50]:
# Update the dictionaries in our tuples of (metadata, wdl task)
for task_tuple in metas_tasks:
    meta = task_tuple[0]
    wdl_task = task_tuple[1]
    additional_meta = parse_meta(wdl_task)
    meta.update(additional_meta)

metas_tasks[:3]
    

[({'tool': 'cnvkit',
   'task': 'create_reference',
   'topic': ['genomics', 'copy_number_variation'],
   'species': ['eukaryote'],
   'operation': ['indexing'],
   'input_sample_data_types': ['data_index',
    'nucleic_acid_sequence_alignment'],
   'input_sample_format_types': ['bam', 'bai'],
   'output_sample_data_types': ['none']},
  'task create_reference {\n  meta {\n    author: "Taylor Firman"\n    email: "tfirman@fredhutch.org"\n    description: "Create CNVkit reference from normal samples or pooled reference"\n    url: "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-cnvkit/ww-cnvkit.wdl"\n    outputs: {\n        reference_cnn: "CNVkit reference file (.cnn)"\n    }\n    topic: "genomics,copy_number_variation"\n    species: "eukaryote"\n    operation: "indexing"\n    input_sample_required: "bam_files:nucleic_acid_sequence_alignment:bam,bam_indices:data_index:bai"\n    input_sample_optional: "target_bed:annotation_track:bed,antitarget_bed:a

## 3. Add WDL tasks and metadata to ChromaDB collection

In [51]:
# Initiate a collection and save to the `data/` folder
chroma_client = chromadb.PersistentClient(path='../data/chroma')
collection = chroma_client.get_or_create_collection(name="wdl_tasks")

In [52]:
# Add 'documents' (wdl tasks)
metadatas, documents= zip(*metas_tasks)
ids = [f'{meta["tool"]}_{meta["task"]}' for meta in metadatas]

collection.add(
    ids=ids,
    documents=list(documents), 
    metadatas=list(metadatas))

## 4. Confirm we can filter by metadata

In [55]:
# Define our filtering criteria
# Must take aligned, indexed data as input and call variants.

my_input_data = ["nucleic_acid_sequence_alignment", "data_index"]
my_input_format = ["bam", "bai"]
my_output_data = ["sequence_variations"]

In [56]:
# Generate filters (will use 'and' to ensure task meets all minimum needs)
filter_input_data = [{"input_sample_data_types": {"$contains": i}} for i in my_input_data]
filter_input_format = [{"input_sample_format_types": {"$contains": i}} for i in my_input_format]
filter_output_data = [{"output_sample_data_types": {"$contains": i}} for i in my_output_data]

# Look at what we'll be filtering on
print(filter_input_data)
print(filter_input_format)
print(filter_output_data)

# Combine
full_filter = filter_input_data + filter_input_format + filter_output_data

[{'input_sample_data_types': {'$contains': 'nucleic_acid_sequence_alignment'}}, {'input_sample_data_types': {'$contains': 'data_index'}}]
[{'input_sample_format_types': {'$contains': 'bam'}}, {'input_sample_format_types': {'$contains': 'bai'}}]
[{'output_sample_data_types': {'$contains': 'sequence_variations'}}]


In [57]:
# Retrieve tasks that meet all criteria
collection.get(where={"$and": full_filter})

{'ids': ['cnvkit_run_cnvkit',
  'delly_delly_call',
  'bcftools_mpileup_call',
  'deepvariant_run_deepvariant',
  'smoove_smoove_call',
  'manta_manta_call',
  'strelka_strelka_germline',
  'strelka_strelka_somatic',
  'gatk_haplotype_caller',
  'gatk_mutect2',
  'gatk_haplotype_caller_parallel',
  'gatk_mutect2_parallel'],
 'embeddings': None,
 'documents': ['task run_cnvkit {\n  meta {\n    author: "Taylor Firman"\n    email: "tfirman@fredhutch.org"\n    description: "Run CNVkit copy number analysis on tumor sample"\n    url: "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-cnvkit/ww-cnvkit.wdl"\n    outputs: {\n        cnv_calls: "CNV calls file (.cns)",\n        cnv_segments: "CNV segments file (.cnr)",\n        cnv_plot: "CNV visualization plot"\n    }\n    topic: "genomics,copy_number_variation"\n    species: "eukaryote"\n    operation: "copy_number_variation_detection"\n    input_sample_required: "tumor_bam:nucleic_acid_sequence_alignment: